# PC Scores
Create and inspect PCA score tables for placenta, cord, and combined datasets.

In [30]:
import os
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path(os.getcwd()).parent.parent))
from src.utils.config import PCA_CSV_DIR, OUTPUTS_DIR, PLACENTA_FILE, CORD_FILE

PC_SCORES_CSV_DIR = os.path.join(PCA_CSV_DIR, "06_pc_scores")
os.makedirs(PC_SCORES_CSV_DIR, exist_ok=True)

print("PCA_CSV_DIR:", PCA_CSV_DIR)
print("PC_SCORES_CSV_DIR:", PC_SCORES_CSV_DIR)
print("PLACENTA_FILE:", PLACENTA_FILE)
print("CORD_FILE:", CORD_FILE)

PCA_CSV_DIR: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv
PC_SCORES_CSV_DIR: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores
PLACENTA_FILE: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/raw/MetaPro_placenta_Jan2022.xlsx
CORD_FILE: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/raw/MetaPro_cord_blood_Jan2022.xlsx


In [4]:
placenta_scores = pd.read_csv(os.path.join(PCA_CSV_DIR, "pca_placenta_only.csv"), index_col="Sample")
cord_scores = pd.read_csv(os.path.join(PCA_CSV_DIR, "pca_cord_only.csv"), index_col="Sample")
combined_scores = pd.read_csv(os.path.join(PCA_CSV_DIR, "pca_combined_placenta_cord.csv"), index_col="Sample")

print("Placenta scores shape:", placenta_scores.shape)
print("Cord scores shape:", cord_scores.shape)
print("Combined scores shape:", combined_scores.shape)

Placenta scores shape: (40, 10)
Cord scores shape: (38, 10)
Combined scores shape: (78, 11)


# ********* PLACENTA **********

# Step 1: Load Placenta Sample Info (Tab 2)

In [5]:

placenta_sample_info = pd.read_excel(PLACENTA_FILE, sheet_name="Sample Info")
print("Placenta Sample Info shape:", placenta_sample_info.shape)
print("Columns:", placenta_sample_info.columns.tolist())
display(placenta_sample_info.head())

Placenta Sample Info shape: (40, 3)
Columns: ['SAMPLE ID', 'Group ID', 'Tube Label']


,SAMPLE ID,Group ID,Tube Label
0,sample-1,A,A20150
1,sample-2,B,A20284
2,sample-3,A,C20159
3,sample-4,B,C20256
4,sample-5,A,C20187


In [31]:
# Step 2: Build Placenta table with Sample ID, PC1 Score, PC2 Score
sample_id_col = next((c for c in placenta_sample_info.columns if c.strip().lower() == "sample id"), None)
if sample_id_col is None:
    raise ValueError("Could not find 'Sample ID' column in Placenta Sample Info sheet")

tube_label_col = next((c for c in placenta_sample_info.columns if "tube" in c.strip().lower()), None)
base_cols = [sample_id_col] + ([tube_label_col] if tube_label_col else [])

placenta_pc12_table = (
    placenta_sample_info[base_cols]
    .rename(columns={sample_id_col: "Sample"})
    .merge(
        placenta_scores[["PC1", "PC2"]].reset_index(),
        on="Sample",
        how="left"
    )
    .rename(columns={"PC1": "PC1 Score", "PC2": "PC2 Score"})
)

print("Placenta PC1/PC2 table shape:", placenta_pc12_table.shape)
print("Columns:", placenta_pc12_table.columns.tolist())
display(placenta_pc12_table.head())

placenta_pc1_2_path = os.path.join(PC_SCORES_CSV_DIR, "placenta_sampleid_pc1_pc2.csv")
placenta_pc12_table.to_csv(placenta_pc1_2_path, index=False)
print("Saved:", placenta_pc1_2_path)

Placenta PC1/PC2 table shape: (40, 4)
Columns: ['Sample', 'Tube Label', 'PC1 Score', 'PC2 Score']


,Sample,Tube Label,PC1 Score,PC2 Score
0,sample-1,A20150,-1.836882,9.674800
1,sample-2,A20284,5.119141,-5.977017
2,sample-3,C20159,5.093578,14.212926
3,sample-4,C20256,-8.885906,6.307899
4,sample-5,C20187,5.511009,5.872692


Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores/placenta_sampleid_pc1_pc2.csv


In [8]:
print("Placenta PC scores (first 5 rows):")
display(placenta_scores.head())

print("Cord PC scores (first 5 rows):")
display(cord_scores.head())

print("Combined PC scores (first 5 rows):")
display(combined_scores.head())

Placenta PC scores (first 5 rows):


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
Sample,,,,,,,,,,
sample-1,-1.836882,9.674800,-11.493513,9.484060,0.099155,2.322369,-1.139683,-1.073316,-3.289946,-5.798883
sample-2,5.119141,-5.977017,-4.488253,-9.709202,5.965110,3.621359,1.531295,0.632809,-3.518376,2.380267
sample-3,5.093578,14.212926,-3.278799,-0.170095,-3.598092,21.238699,-2.550390,-0.526630,2.497083,-3.838249
sample-4,-8.885906,6.307899,-7.511825,-2.450200,1.929065,-3.779747,8.049089,2.565184,-5.789312,-3.894546
sample-5,5.511009,5.872692,-22.042687,5.401725,0.019815,-7.527623,-4.545568,-3.112662,2.277956,-1.828105


Cord PC scores (first 5 rows):


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
Sample,,,,,,,,,,
sample-1,-3.121207,2.049936,1.754605,1.398588,-11.495152,1.356310,-2.605005,1.662604,-6.628719,-6.049199
sample-2,-4.443159,2.927280,0.969894,2.652615,-10.920971,-5.223435,-3.559969,-3.773566,4.750779,6.434946
sample-3,-5.233151,-2.264805,3.269364,5.298122,-9.306952,4.879907,0.309220,4.208251,-3.432269,0.978891
sample-4,-3.450644,-1.853362,5.576497,-9.451389,-6.956448,0.867721,-1.079095,2.294198,6.855306,6.947264
sample-5,-11.465463,-4.958626,-4.972062,1.145932,-1.643855,2.081935,1.300950,1.117204,-3.436802,9.226688


Combined PC scores (first 5 rows):


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,Group
Sample,,,,,,,,,,,
sample-1,-2.756424,2.235432,-4.353985,-1.815985,-3.662753,-6.218987,3.882894,10.254538,1.188619,-2.048325,Placenta
sample-2,1.886773,-1.920130,2.731777,2.405389,-2.297303,-0.876648,-7.072831,3.249841,-0.120189,5.423795,Placenta
sample-3,4.606224,7.816481,-8.664037,-7.884191,-2.404590,6.018448,-2.966429,7.831717,-1.174551,-3.184035,Placenta
sample-4,-10.042947,-2.423579,-4.087951,-2.286382,-1.926122,-4.806377,-1.502296,5.306955,0.225396,5.076425,Placenta
sample-5,2.250327,0.227796,1.991520,-4.315641,-1.550212,-15.778812,2.117171,10.188628,1.794733,-1.603042,Placenta


In [19]:
# Step 3: Load mapping workbook and use IDCode as key
project_root = Path(os.getcwd()).parent.parent
candidate_paths = [
    project_root / "src" / "raw" / "mata2022_caco_mar242022.xlsx",
    project_root / "other" / "mata2022_caco_mar242022.xlsx",
]
mapping_file = next((p for p in candidate_paths if p.exists()), None)
if mapping_file is None:
    raise FileNotFoundError("Could not find mata2022_caco_mar242022.xlsx in src/raw or other")

xls = pd.ExcelFile(mapping_file)
target_sheet = xls.sheet_names[0]
mapping_df = pd.read_excel(mapping_file, sheet_name=target_sheet)

print("Mapping workbook:", mapping_file)
print("Selected sheet:", target_sheet)
print("Mapping shape:", mapping_df.shape)
print("Mapping columns:", mapping_df.columns.tolist())

def normalize_col(col_name):
    return str(col_name).strip().lower().replace(" ", "").replace("_", "")

idcode_col = next((c for c in mapping_df.columns if normalize_col(c) == "idcode"), None)
if idcode_col is None:
    raise ValueError("Could not find 'IDCode' column in mapping workbook")

print(f"Using mapping key column: {idcode_col} (IDCode)")
display(mapping_df.head())

Mapping workbook: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/raw/mata2022_caco_mar242022.xlsx
Selected sheet: mata2022_caco_mar242022
Mapping shape: (40, 156)
Mapping columns: ['IDCode', 'matchid', 'placenta', 'cord', 'GDM', 'PIH', 'Ma_age', 'Ma_ethnic', 'Ma_edu', 'GA_calculated', 'Sex_Male_1', 'birthweight', 'lbw', 'preterm', 'sga', 'LGA', 'prom', 'pre_BMI_new', 'pre_BMI_Cat_new', 'Ma_age_3grp', 'BWZScore', 'A5_HHINCOME', 'Fmincome', 'Fmincome2', 'Fa_age', 'B3_GDM', 'B3_DM', 'B3_HBP', 'C1_AGEMENS', 'nulliparity', 'D2_FDM', 'D2_FGDM', 'D2_FGHTN', 'H1_SMOKER', 'H8_LIVESMKR', 'I1_ALCOHOL', 'O1_JOBACTIVITY', 'O2_PHYSACTIV', 'wkly_PM25_hmwk_12', 'wkly_PM25_hmwk_11', 'wkly_PM25_hmwk_10', 'wkly_PM25_hmwk_9', 'wkly_PM25_hmwk_8', 'wkly_PM25_hmwk_7', 'wkly_PM25_hmwk_6', 'wkly_PM25_hmwk_5', 'wkly_PM25_hmwk_4', 'wkly_PM25_hmwk_3', 'wkly_PM25_hmwk_2', 'wkly_PM25_hmwk_1', 'wkly_PM25_hmwk_0', 'wkly_PM25_hmwk1', 'wkly_PM25_

,IDCode,matchid,placenta,cord,GDM,PIH,Ma_age,Ma_ethnic,Ma_edu,GA_calculated,...,p3_OTHER_VEGETABLE2,p3_NUTS2,p3_FTUIT2,p3_ONION2,p3_RAW_GARLIC2,p3_COOKED_GARLIC2,diet_protein,diet_vegfruit,diet_nuts,diet_alliaceous
0,20150,1,9,0,1,0,38,1,2.0,40,...,5.5,3.50,1.5,0.65,NaN,NaN,6.15,17.0,3.50,0.65
1,20159,2,1,1,1,0,28,2,2.0,40,...,1.5,1.50,1.5,1.50,0.0,0.0,3.25,4.5,1.50,1.50
2,20184,19,9,1,0,0,37,1,2.0,38,...,10.0,10.00,7.0,7.00,NaN,NaN,3.00,24.0,10.00,7.00
3,20186,17,1,1,1,0,38,1,2.0,42,...,7.0,7.00,3.5,0.00,NaN,NaN,17.00,17.5,7.00,0.00
4,20187,3,1,1,1,0,37,1,2.0,43,...,10.0,0.25,7.0,7.00,NaN,NaN,8.50,24.0,0.25,7.00


In [32]:
# Step 4: Map PC1/PC2 using IDCode <-> Tube Label numeric code
if 'Tube Label' not in placenta_pc12_table.columns:
    raise ValueError("Tube Label column is required for mapping to IDCode")

pc12_for_map = placenta_pc12_table.copy()
pc12_for_map['TubeCode_key'] = pc12_for_map['Tube Label'].astype(str).str.extract(r'(\d+)')[0]
pc12_for_map = pc12_for_map[['TubeCode_key', 'Sample', 'PC1 Score', 'PC2 Score']].drop_duplicates(subset=['TubeCode_key'])

mapped_df = mapping_df.copy()
idcode_numeric = mapped_df[idcode_col].astype(str).str.extract(r'(\d+)')[0]

mapped_with_pc = mapped_df.merge(
    pc12_for_map,
    left_on=idcode_numeric,
    right_on='TubeCode_key',
    how='left'
 )

insert_cols = ['Sample', 'PC1 Score', 'PC2 Score', 'TubeCode_key']
remaining_cols = [c for c in mapped_with_pc.columns if c not in insert_cols]
id_idx = remaining_cols.index(idcode_col)
ordered_cols = remaining_cols[:id_idx] + insert_cols + remaining_cols[id_idx:]
mapped_with_pc = mapped_with_pc[ordered_cols]

matched_n = mapped_with_pc['PC1 Score'].notna().sum()
print("Mapped table shape:", mapped_with_pc.shape)
print("Matched rows with PC1/PC2:", matched_n)
print("Unmatched rows:", mapped_with_pc.shape[0] - matched_n)

display(mapped_with_pc.head())

mapped_csv_path = os.path.join(PC_SCORES_CSV_DIR, "placenta_mapped_with_pc1_pc2.csv")
mapped_xlsx_path = os.path.join(PC_SCORES_CSV_DIR, "placenta_mapped_with_pc1_pc2.xlsx")
mapped_with_pc.to_csv(mapped_csv_path, index=False)
mapped_with_pc.to_excel(mapped_xlsx_path, index=False)

print("Saved:", mapped_csv_path)
print("Saved:", mapped_xlsx_path)

Mapped table shape: (40, 160)
Matched rows with PC1/PC2: 40
Unmatched rows: 0


,Sample,PC1 Score,PC2 Score,TubeCode_key,IDCode,matchid,placenta,cord,GDM,PIH,...,p3_OTHER_VEGETABLE2,p3_NUTS2,p3_FTUIT2,p3_ONION2,p3_RAW_GARLIC2,p3_COOKED_GARLIC2,diet_protein,diet_vegfruit,diet_nuts,diet_alliaceous
0,sample-1,-1.836882,9.674800,20150,20150,1,9,0,1,0,...,5.5,3.50,1.5,0.65,NaN,NaN,6.15,17.0,3.50,0.65
1,sample-3,5.093578,14.212926,20159,20159,2,1,1,1,0,...,1.5,1.50,1.5,1.50,0.0,0.0,3.25,4.5,1.50,1.50
2,sample-38,-1.352176,13.133986,20184,20184,19,9,1,0,0,...,10.0,10.00,7.0,7.00,NaN,NaN,3.00,24.0,10.00,7.00
3,sample-33,-5.426715,15.072755,20186,20186,17,1,1,1,0,...,7.0,7.00,3.5,0.00,NaN,NaN,17.00,17.5,7.00,0.00
4,sample-5,5.511009,5.872692,20187,20187,3,1,1,1,0,...,10.0,0.25,7.0,7.00,NaN,NaN,8.50,24.0,0.25,7.00


Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores/placenta_mapped_with_pc1_pc2.csv
Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores/placenta_mapped_with_pc1_pc2.xlsx


# ********* CHORD **********

In [22]:
# Step 1 (Cord): Load Cord Sample Info
cord_sample_info = pd.read_excel(CORD_FILE, sheet_name="Sample Info")
print("Cord Sample Info shape:", cord_sample_info.shape)
print("Columns:", cord_sample_info.columns.tolist())
display(cord_sample_info.head())

Cord Sample Info shape: (38, 3)
Columns: ['SAMPLE ID', 'Group ID', 'Tube Label']


,SAMPLE ID,Group ID,Tube Label
0,sample-1,B,20284S1
1,sample-2,A,20159S1
2,sample-3,B,20256S1
3,sample-4,A,20187S1
4,sample-5,B,20219S1


In [33]:
# Step 2 (Cord): Build table with Sample, Tube Label, PC1 Score, PC2 Score
cord_sample_id_col = next((c for c in cord_sample_info.columns if c.strip().lower() == "sample id"), None)
if cord_sample_id_col is None:
    raise ValueError("Could not find 'Sample ID' column in Cord Sample Info sheet")

cord_tube_label_col = next((c for c in cord_sample_info.columns if "tube" in c.strip().lower()), None)
cord_base_cols = [cord_sample_id_col] + ([cord_tube_label_col] if cord_tube_label_col else [])

cord_pc12_table = (
    cord_sample_info[cord_base_cols]
    .rename(columns={cord_sample_id_col: "Sample"})
    .merge(
        cord_scores[["PC1", "PC2"]].reset_index(),
        on="Sample",
        how="left"
    )
    .rename(columns={"PC1": "PC1 Score", "PC2": "PC2 Score"})
)

print("Cord PC1/PC2 table shape:", cord_pc12_table.shape)
print("Columns:", cord_pc12_table.columns.tolist())
display(cord_pc12_table.head())

cord_pc1_2_path = os.path.join(PC_SCORES_CSV_DIR, "cord_sampleid_pc1_pc2.csv")
cord_pc12_table.to_csv(cord_pc1_2_path, index=False)
print("Saved:", cord_pc1_2_path)

Cord PC1/PC2 table shape: (38, 4)
Columns: ['Sample', 'Tube Label', 'PC1 Score', 'PC2 Score']


,Sample,Tube Label,PC1 Score,PC2 Score
0,sample-1,20284S1,-3.121207,2.049936
1,sample-2,20159S1,-4.443159,2.927280
2,sample-3,20256S1,-5.233151,-2.264805
3,sample-4,20187S1,-3.450644,-1.853362
4,sample-5,20219S1,-11.465463,-4.958626


Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores/cord_sampleid_pc1_pc2.csv


In [34]:
# Step 3 (Cord): Map PC1/PC2 using IDCode <-> Tube Label numeric code
if 'Tube Label' not in cord_pc12_table.columns:
    raise ValueError("Tube Label column is required for mapping to IDCode")

cord_pc12_for_map = cord_pc12_table.copy()
cord_pc12_for_map['TubeCode_key'] = cord_pc12_for_map['Tube Label'].astype(str).str.extract(r'(\d+)')[0]
cord_pc12_for_map = cord_pc12_for_map[['TubeCode_key', 'Sample', 'PC1 Score', 'PC2 Score']].drop_duplicates(subset=['TubeCode_key'])

cord_mapped_df = mapping_df.copy()
cord_idcode_numeric = cord_mapped_df[idcode_col].astype(str).str.extract(r'(\d+)')[0]

cord_mapped_with_pc = cord_mapped_df.merge(
    cord_pc12_for_map,
    left_on=cord_idcode_numeric,
    right_on='TubeCode_key',
    how='left'
 )

cord_insert_cols = ['Sample', 'PC1 Score', 'PC2 Score', 'TubeCode_key']
cord_remaining_cols = [c for c in cord_mapped_with_pc.columns if c not in cord_insert_cols]
cord_id_idx = cord_remaining_cols.index(idcode_col)
cord_ordered_cols = cord_remaining_cols[:cord_id_idx] + cord_insert_cols + cord_remaining_cols[cord_id_idx:]
cord_mapped_with_pc = cord_mapped_with_pc[cord_ordered_cols]

cord_matched_n = cord_mapped_with_pc['PC1 Score'].notna().sum()
print("Cord mapped table shape:", cord_mapped_with_pc.shape)
print("Cord matched rows with PC1/PC2:", cord_matched_n)
print("Cord unmatched rows:", cord_mapped_with_pc.shape[0] - cord_matched_n)

display(cord_mapped_with_pc.head())

cord_mapped_csv_path = os.path.join(PC_SCORES_CSV_DIR, "cord_mapped_with_pc1_pc2.csv")
cord_mapped_xlsx_path = os.path.join(PC_SCORES_CSV_DIR, "cord_mapped_with_pc1_pc2.xlsx")
cord_mapped_with_pc.to_csv(cord_mapped_csv_path, index=False)
cord_mapped_with_pc.to_excel(cord_mapped_xlsx_path, index=False)

print("Saved:", cord_mapped_csv_path)
print("Saved:", cord_mapped_xlsx_path)

Cord mapped table shape: (40, 160)
Cord matched rows with PC1/PC2: 38
Cord unmatched rows: 2


,Sample,PC1 Score,PC2 Score,TubeCode_key,IDCode,matchid,placenta,cord,GDM,PIH,...,p3_OTHER_VEGETABLE2,p3_NUTS2,p3_FTUIT2,p3_ONION2,p3_RAW_GARLIC2,p3_COOKED_GARLIC2,diet_protein,diet_vegfruit,diet_nuts,diet_alliaceous
0,NaN,NaN,NaN,20150,20150,1,9,0,1,0,...,5.5,3.50,1.5,0.65,NaN,NaN,6.15,17.0,3.50,0.65
1,sample-2,-4.443159,2.927280,20159,20159,2,1,1,1,0,...,1.5,1.50,1.5,1.50,0.0,0.0,3.25,4.5,1.50,1.50
2,sample-36,-9.995735,-7.532694,20184,20184,19,9,1,0,0,...,10.0,10.00,7.0,7.00,NaN,NaN,3.00,24.0,10.00,7.00
3,sample-31,-3.362262,5.199605,20186,20186,17,1,1,1,0,...,7.0,7.00,3.5,0.00,NaN,NaN,17.00,17.5,7.00,0.00
4,sample-4,-3.450644,-1.853362,20187,20187,3,1,1,1,0,...,10.0,0.25,7.0,7.00,NaN,NaN,8.50,24.0,0.25,7.00


Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores/cord_mapped_with_pc1_pc2.csv
Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores/cord_mapped_with_pc1_pc2.xlsx


In [25]:
# Step 4 (Cord): List unmatched IDCode values
unmatched_cord = cord_mapped_with_pc[cord_mapped_with_pc['PC1 Score'].isna()].copy()
unmatched_cols = [idcode_col] + (["TubeCode_key"] if "TubeCode_key" in unmatched_cord.columns else [])
unmatched_cord = unmatched_cord[unmatched_cols]

print("Unmatched cord rows:", unmatched_cord.shape[0])
print("Unmatched IDCode values:", unmatched_cord[idcode_col].dropna().astype(str).tolist())
display(unmatched_cord)

Unmatched cord rows: 2
Unmatched IDCode values: ['20150', '20194']


,IDCode,TubeCode_key
0,20150,20150
6,20194,20194


In [26]:
# Step 5 (Cord): Explain why unmatched IDCodes did not map
if unmatched_cord.empty:
    print("No unmatched cord IDCodes.")
else:
    cord_sample_id_col = next((c for c in cord_sample_info.columns if c.strip().lower() == "sample id"), None)
    if cord_sample_id_col is None:
        raise ValueError("Could not find 'Sample ID' column in Cord Sample Info sheet")

    cord_info_keys = cord_sample_info[[cord_sample_id_col, 'Tube Label']].copy()
    cord_info_keys['TubeCode_key'] = cord_info_keys['Tube Label'].astype(str).str.extract(r'(\d+)')[0]

    pca_sample_set = set(cord_scores.index.astype(str))
    diag_rows = []
    for raw_id in unmatched_cord[idcode_col].dropna().astype(str):
        id_num = ''.join(ch for ch in raw_id if ch.isdigit())
        hit = cord_info_keys[cord_info_keys['TubeCode_key'] == id_num]

        if hit.empty:
            matched_sample = None
            has_pca = False
            reason = "IDCode not present in cord Sample Info Tube Label"
        else:
            matched_sample = str(hit.iloc[0][cord_sample_id_col])
            has_pca = matched_sample in pca_sample_set
            reason = "Sample exists but missing PCA score" if not has_pca else "Mapped candidate exists (recheck Step 3 merge)"

        diag_rows.append({
            'IDCode': raw_id,
            'TubeCode_key': id_num,
            'Matched Sample ID (cord info)': matched_sample,
            'Sample has PCA score': has_pca,
            'Reason': reason
        })

    unmatched_reason_df = pd.DataFrame(diag_rows)
    display(unmatched_reason_df)

,IDCode,TubeCode_key,Matched Sample ID (cord info),Sample has PCA score,Reason
0,20150,20150,None,False,IDCode not present in cord Sample Info Tube Label
1,20194,20194,None,False,IDCode not present in cord Sample Info Tube Label


# ********* COMMON (PLACENTA + CORD) **********

In [27]:
# Step 1 (Common): Build unified sample info with Group
placenta_info_common = placenta_sample_info.copy()
placenta_info_sample_col = next((c for c in placenta_info_common.columns if c.strip().lower() == "sample id"), None)
if placenta_info_sample_col is None:
    raise ValueError("Could not find 'Sample ID' column in Placenta Sample Info sheet")
placenta_info_tube_col = next((c for c in placenta_info_common.columns if "tube" in c.strip().lower()), None)
if placenta_info_tube_col is None:
    raise ValueError("Could not find Tube Label column in Placenta Sample Info sheet")

placenta_info_common = placenta_info_common[[placenta_info_sample_col, placenta_info_tube_col]].rename(
    columns={placenta_info_sample_col: 'Sample', placenta_info_tube_col: 'Tube Label'}
)
placenta_info_common['Group'] = 'Placenta'

cord_info_common = cord_sample_info.copy()
cord_info_sample_col = next((c for c in cord_info_common.columns if c.strip().lower() == "sample id"), None)
if cord_info_sample_col is None:
    raise ValueError("Could not find 'Sample ID' column in Cord Sample Info sheet")
cord_info_tube_col = next((c for c in cord_info_common.columns if "tube" in c.strip().lower()), None)
if cord_info_tube_col is None:
    raise ValueError("Could not find Tube Label column in Cord Sample Info sheet")

cord_info_common = cord_info_common[[cord_info_sample_col, cord_info_tube_col]].rename(
    columns={cord_info_sample_col: 'Sample', cord_info_tube_col: 'Tube Label'}
)
cord_info_common['Group'] = 'Cord'

common_sample_info = pd.concat([placenta_info_common, cord_info_common], ignore_index=True)
print("Common sample info shape:", common_sample_info.shape)
display(common_sample_info.head())

Common sample info shape: (78, 3)


,Sample,Tube Label,Group
0,sample-1,A20150,Placenta
1,sample-2,A20284,Placenta
2,sample-3,C20159,Placenta
3,sample-4,C20256,Placenta
4,sample-5,C20187,Placenta


In [35]:
# Step 2 (Common): Build combined table with Sample, Group, Tube Label, PC1 Score, PC2 Score
combined_pc12_table = (
    combined_scores.reset_index()[['Sample', 'Group', 'PC1', 'PC2']]
    .rename(columns={'PC1': 'PC1 Score', 'PC2': 'PC2 Score'})
    .merge(common_sample_info, on=['Sample', 'Group'], how='left')
)
combined_pc12_table = combined_pc12_table[['Sample', 'Group', 'Tube Label', 'PC1 Score', 'PC2 Score']]

print("Combined PC1/PC2 table shape:", combined_pc12_table.shape)
print("Rows missing Tube Label:", combined_pc12_table['Tube Label'].isna().sum())
display(combined_pc12_table.head())

combined_pc1_2_path = os.path.join(PC_SCORES_CSV_DIR, "common_sampleid_group_pc1_pc2.csv")
combined_pc12_table.to_csv(combined_pc1_2_path, index=False)
print("Saved:", combined_pc1_2_path)

Combined PC1/PC2 table shape: (78, 5)
Rows missing Tube Label: 0


,Sample,Group,Tube Label,PC1 Score,PC2 Score
0,sample-1,Placenta,A20150,-2.756424,2.235432
1,sample-2,Placenta,A20284,1.886773,-1.920130
2,sample-3,Placenta,C20159,4.606224,7.816481
3,sample-4,Placenta,C20256,-10.042947,-2.423579
4,sample-5,Placenta,C20187,2.250327,0.227796


Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores/common_sampleid_group_pc1_pc2.csv


In [36]:
# Step 3 (Common): Map combined PC1/PC2 using IDCode <-> Tube Label numeric code
if 'Tube Label' not in combined_pc12_table.columns:
    raise ValueError("Tube Label column is required for mapping to IDCode")

common_pc12_for_map = combined_pc12_table.copy()
common_pc12_for_map['TubeCode_key'] = common_pc12_for_map['Tube Label'].astype(str).str.extract(r'(\d+)')[0]
common_pc12_for_map = common_pc12_for_map[['TubeCode_key', 'Sample', 'Group', 'PC1 Score', 'PC2 Score']]
common_pc12_for_map = common_pc12_for_map.drop_duplicates(subset=['TubeCode_key', 'Group'])

common_mapped_df = mapping_df.copy()
common_mapped_df['IDCode_key'] = common_mapped_df[idcode_col].astype(str).str.extract(r'(\d+)')[0]

common_mapped_with_pc = common_mapped_df.merge(
    common_pc12_for_map,
    left_on='IDCode_key',
    right_on='TubeCode_key',
    how='left'
 )

common_insert_cols = ['Sample', 'Group', 'PC1 Score', 'PC2 Score', 'TubeCode_key']
common_remaining_cols = [c for c in common_mapped_with_pc.columns if c not in common_insert_cols + ['IDCode_key']]
common_id_idx = common_remaining_cols.index(idcode_col)
common_ordered_cols = common_remaining_cols[:common_id_idx] + common_insert_cols + common_remaining_cols[common_id_idx:]
common_mapped_with_pc = common_mapped_with_pc[common_ordered_cols]

common_matched_n = common_mapped_with_pc['PC1 Score'].notna().sum()
print("Common mapped table shape:", common_mapped_with_pc.shape)
print("Common matched rows with PC1/PC2:", common_matched_n)
print("Common unmatched rows:", common_mapped_with_pc.shape[0] - common_matched_n)

display(common_mapped_with_pc.head())

common_mapped_csv_path = os.path.join(PC_SCORES_CSV_DIR, "common_mapped_with_pc1_pc2.csv")
common_mapped_xlsx_path = os.path.join(PC_SCORES_CSV_DIR, "common_mapped_with_pc1_pc2.xlsx")
common_mapped_with_pc.to_csv(common_mapped_csv_path, index=False)
common_mapped_with_pc.to_excel(common_mapped_xlsx_path, index=False)

print("Saved:", common_mapped_csv_path)
print("Saved:", common_mapped_xlsx_path)

Common mapped table shape: (78, 161)
Common matched rows with PC1/PC2: 78
Common unmatched rows: 0


,Sample,Group,PC1 Score,PC2 Score,TubeCode_key,IDCode,matchid,placenta,cord,GDM,...,p3_OTHER_VEGETABLE2,p3_NUTS2,p3_FTUIT2,p3_ONION2,p3_RAW_GARLIC2,p3_COOKED_GARLIC2,diet_protein,diet_vegfruit,diet_nuts,diet_alliaceous
0,sample-1,Placenta,-2.756424,2.235432,20150,20150,1,9,0,1,...,5.5,3.5,1.5,0.65,NaN,NaN,6.15,17.0,3.5,0.65
1,sample-3,Placenta,4.606224,7.816481,20159,20159,2,1,1,1,...,1.5,1.5,1.5,1.50,0.0,0.0,3.25,4.5,1.5,1.50
2,sample-2,Cord,-0.674444,4.061088,20159,20159,2,1,1,1,...,1.5,1.5,1.5,1.50,0.0,0.0,3.25,4.5,1.5,1.50
3,sample-38,Placenta,-3.168824,7.905406,20184,20184,19,9,1,0,...,10.0,10.0,7.0,7.00,NaN,NaN,3.00,24.0,10.0,7.00
4,sample-36,Cord,-9.580509,1.782245,20184,20184,19,9,1,0,...,10.0,10.0,7.0,7.00,NaN,NaN,3.00,24.0,10.0,7.00


Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores/common_mapped_with_pc1_pc2.csv
Saved: /Users/aishvarya/Library/CloudStorage/OneDrive-UniversityatBuffalo/Desktop/Project/Metabolomics/Metabolomics_Project/src/../outputs/pca/output_csv/06_pc_scores/common_mapped_with_pc1_pc2.xlsx
